<a href="https://colab.research.google.com/github/Viv-Dave/deep-learning/blob/main/mars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("brsdincer/hirise-map-mars-nasa-image")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/brsdincer/hirise-map-mars-nasa-image/versions/1


In [7]:
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
# /root/.cache/kagglehub/datasets/brsdincer/hirise-map-mars-nasa-image
dataset_path = '/root/.cache/kagglehub/datasets/brsdincer/hirise-map-mars-nasa-image/versions/1/hirise-map-proj-v3'
# csv_path = '/root/.cache/kagglehub/datasets/brsdincer/hirise-map-mars-nasa-image/versions/1/hirise-map-proj-v3/landmarks_map-proj-v3_classmap.csv'
class MarsImageDataset(Dataset):
  def __init__(self,root_dir,transform=None):
    self.root_dir = root_dir
    self.transform = transform
    # /kaggle/input/hirise-map-mars-nasa-image/hirise-map-proj-v3/map-proj-v3
    self.image_dir = os.path.join(root_dir, 'map-proj-v3')
    # self.txt_dir = os.path.join(root_dir, '/labels-map-proj-v3', 'txt')
    self.txt_dir = pd.read_csv(root_dir+'/labels-map-proj-v3.txt',sep='\s+')
    self.csv_dir = pd.read_csv(root_dir+'/landmarks_map-proj-v3_classmap.csv')
  def __len__(self):
    return len(self.txt_dir)

  def __getitem__(self, index):
    img_name = self.txt_dir.iloc[index, 0]
    original_label = self.txt_dir.iloc[index, 1]
    img_path = os.path.join(self.image_dir, img_name)
    image = Image.open(img_path).convert("RGB")

    if self.transform:
            image = self.transform(image)

    return image, original_label

In [8]:
from torchvision import transforms
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
mars_dataset = MarsImageDataset(root_dir=dataset_path, transform = transform)
train_indices, val_indices = train_test_split(list(range(len(mars_dataset))),test_size=0.2, random_state=42)
train_dataset = Subset(mars_dataset, train_indices)
val_dataset = Subset(mars_dataset, val_indices)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)
print(mars_dataset.txt_dir.iloc[:, 1].unique())

[0 5 1 3 2 6 7 4]


In [9]:
import torch.nn as nn
import torch.nn.functional as F
import torch
class Classifier(nn.Module):
  def __init__(self):
    super(Classifier, self).__init__()
    self.conv1 = nn.Conv2d(in_channels=3,out_channels=8,kernel_size=3, padding=1) #224*224
    self.bn1 = nn.BatchNorm2d(8)

    self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1) #112*112
    self.bn2 = nn.BatchNorm2d(16)

    self.conv3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1) #56*56
    self.bn3 = nn.BatchNorm2d(32)

    self.conv4 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1) #28*28
    self.bn4 = nn.BatchNorm2d(64)

    self.conv5 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1) #14*14
    self.bn5 = nn.BatchNorm2d(128)

    self.conv6 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1) #7*7
    self.bn6 = nn.BatchNorm2d(256)

    self.pool = nn.MaxPool2d(2,2)

    self.fc1 = nn.Linear(256*3*3, 1152)
    self.fc_bn1 = nn.BatchNorm1d(1152)
    self.fc2 = nn.Linear(1152, 288)
    self.fc_bn2 = nn.BatchNorm1d(288)
    self.fc3 = nn.Linear(288, 8)

  def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x))))
        x = self.pool(F.relu(self.bn6(self.conv6(x))))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc_bn1(self.fc1(x)))
        x = F.relu(self.fc_bn2(self.fc2(x)))
        x = self.fc3(x)
        return x
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [10]:
import torch.optim as optim
model = Classifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [19]:
EPOCH = 10
for epoch in range(EPOCH):
    running_loss = 0.0
    for i, data in enumerate(train_loader,0):
      input, labels =  data[0].to(device), data[1].to(device)

      optimizer.zero_grad()

      output = model(input)
      loss = criterion(output, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      if (i + 1) % 100 == 0:
          print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}')
          running_loss = 0.0
print('Finished Training')
#Training Done on 5 EPOCHS, FINAL LOSS: 0.068
#Training Done on 10 EPOCHS, FINAL LOSS:

[1,   100] loss: 0.242
[1,   200] loss: 0.172
[1,   300] loss: 0.099
[1,   400] loss: 0.122
[1,   500] loss: 0.116
[1,   600] loss: 0.074
[1,   700] loss: 0.085
[1,   800] loss: 0.091
[1,   900] loss: 0.088
[1,  1000] loss: 0.104
[1,  1100] loss: 0.103
[1,  1200] loss: 0.088
[1,  1300] loss: 0.087
[1,  1400] loss: 0.057
[1,  1500] loss: 0.072
[1,  1600] loss: 0.106
[1,  1700] loss: 0.074
[1,  1800] loss: 0.101
[2,   100] loss: 0.041
[2,   200] loss: 0.040
[2,   300] loss: 0.048
[2,   400] loss: 0.063
[2,   500] loss: 0.045
[2,   600] loss: 0.057
[2,   700] loss: 0.070
[2,   800] loss: 0.057
[2,   900] loss: 0.056
[2,  1000] loss: 0.042
[2,  1100] loss: 0.073
[2,  1200] loss: 0.073
[2,  1300] loss: 0.056
[2,  1400] loss: 0.076
[2,  1500] loss: 0.068
[2,  1600] loss: 0.057
[2,  1700] loss: 0.068
[2,  1800] loss: 0.055
[3,   100] loss: 0.045
[3,   200] loss: 0.028
[3,   300] loss: 0.040
[3,   400] loss: 0.039
[3,   500] loss: 0.052
[3,   600] loss: 0.051
[3,   700] loss: 0.053
[3,   800] 

In [21]:
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/MARS2.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved to: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to: /content/drive/MyDrive/MARS2.pth


In [22]:
#Evaluating Mode baby!
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in test_loader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
accuracy = 100 * correct / total
print(f'Accuracy of the network on the {total} test images: {accuracy:.2f} %')

Accuracy of the network on the 14606 test images: 92.44 %


In [ ]:
csv_dir = pd.read_csv(dataset_path+'/landmarks_map-proj-v3_classmap.csv')
print(csv_dir.head(10))
labels_df = pd.read_csv(dataset_path+'/labels-map-proj-v3.txt')
print(labels_df.tail(10))